In [ ]:
/**
 * Synchronous & Asynchronous Patterns:
 * > Now that we know about scalings and caching, let's understand the most important 
 *   thing that makes your system robust and scale to millions of users.
 *   - The Sync and Async Programming Patterns
 *   - Async Patterns in Microservice Architecture
 *   - Async Processing Patterns: queues, pub/sub, and event-driven designs.
 *   - Message Queues : single vs distributed queues, priority handling.
 *   - Pub/Sub vs Event Driven vs Queues : When to use what?
 *   - Fan-Out Architectures: scaling event-driven systems.
 *   - CQRS - Command Query Responsibility Segregation
*/

In [ ]:
/**
 * The Sync and Async Programming Patterns:
 * - Programming m humare paas 2 patterns hote hai jiske saath hm code
 *   karte hai:
 *   a. Synchronous
 *   b. Asynchronous
 * - Sync Pattern m hum tasks ko line by line krte hai. It's kind of
 *   blocking task because bich k kisi process ne jyada time liya to
 *   niche k saare process blocked ho jaenge, because each task is 
 *   dependent on previous task.
 * - Async Pattern m hm kisi task ko background m offload kr skte hai
 *   and we can continue to do other task aur jab ye result ready ho
 *   jaega then we will do some continuation of the task.
 * 
 * Example:
 * - Let's say we are building any kind of application (CRUD), to uske
 *   andar ek Signup hoga. Iss feature ka pseudo code kaisa dikhega?
 * - User apne credentials submit karega, hum sbse pehle isko:
 *   > rate limiting krnge,
 *   > validate karnge,
 *   > Password Hash 
 *   > DB insertion
 *   > Email
 *   > Return Token
 * - Now this code absolutely works fine but there is a problem.
 *   "What if we get 1M calls?". Jisse mera system bottleneck banega
 *   aur hang krnge lagega.
 * - Rate Limiting ko hm background m run kr skte hai? No! Because
 *   jitni der m rate limiting check kr k aaega utne der m hm task kr
 *   chuke honge. So, we have to do it in a synchronous way.
 * - User ne id, password diya for validation. Kya validation ko 
 *   background m kr de? No!
 * - Password Hashing Background m kr do? No!
 * - Database Inseration background m kr de? No! Agar database insertion
 *   hi nhi hua to Signup ka matlab hi kya rha.
 * - Email ko background kr de? Yes! Email ko yha pe rakhne se multiple
 *   problems ho rhi hai:
 *   a. Ye meri latency increase kr rha hai, because this is dependent
 *      on some external system. Ho sakta hai ki hm yha pe gmail ka server
 *      use kr rhe hai, ya fir ho sakta hai hm koi proton email use
 *      kr rhe hai, etc. Because email send krne m ek latency aaegi, means
 *      time lagega because it's a third party service. Aur agar yha pe
 *      hm 1M calls krnge to ho sakta hai ki my system is good enough
 *      to get 1M calls but ye external system uss time pe unavailable
 *      ho, aur agar ye system fail kr rha hai to mera ye code fail kr
 *      jaega aur hm user ko kabhi token return nhi kr paenge. Aur agar
 *      yha pe koi external rate limiting lagi hui hai to wo v ek bottleneck
 *      banegi because I don't want to send 1M emails. To isko hm
 *      background kr skte hai.
 * - Kya token return hm background kr skte hai? No!
 *   So, we have to do only for email, but how to do background?
 * 
 * Solution: Background Service
 * - Utlimately everything is machine. Let's say EC2 is our API Machine
 *   jiske andar ye saara code ho rha hai. Ab mai chahta hu ki API machine
 *   should not be responsible for sending the email.
 * - Email k liye hum alag se machine banana chahte hai jo inn sab chijo
 *   ko handle kare background k andar.
 * - Aur background m krne k liye humein ek simple server lagana hai 
 *   jiske andar kuch random code hoga jo email send karega, agar koi
 *   failure hota hai to uska retry v yahi sambhalega, rate limiting v
 *   karega.
 * 
 * Problem: How will two microservices communicate with each other?
 * - API call should be HTTP call to Email Service, and ye email
 *   service iss HTTP call ko accept karegi lekin synchronously forward
 *   nhi karegi. Ho sakta hai ye apne kisi buffer, array, linkedlist 
 *   m usko store kre legi, and based on some frequent interval ye 
 *   response send kr dega. This is a kind of async way.
 * - No need to go for queues directly when we can solve easy way,
 *   because hum request ko linked list k head m store krte hai aur
 *   response tail se remove krte hai to wo v ek tarah ka queue ho gya.
 *   Similarly, stack v use kr skte hai. And this is an example of
 *   asynchronous background processing/pattern.
 * 
 * Problem: What if the email service goes down?
 * - Agar email service goes down then humari http calls fail hone lag
 *   jaegi jisse dobara se mera email wala kaam khatm ho jaega.
 * 
 * Solution: 
 * - Humne Single Point of Failure hata diya aur multiple servers
 *   rakh diye, and it's very rare ki teeno service down chali jae.
 * - That means humne iske aage ek load balancer laga diya (nginx).
 * 
 *                                             +----+
 *                                        +--->|    |
 *                                        |    +----+
 *                                        |
 *   +----+ <-------------- +-------+     |    +----+
 *   |    |                 | Nginx |-----+--->|    |
 *   +----+ --------------> +-------+     |    +----+
 * API Machine   https                    |
 *                                        |    +----+
 *                                        +--->|    |
 *                                             +----+
 *                                          Email Machine
 * 
 * - Teeno k andar LinkedList chal rha hai, aur teeno kisi external
 *   server k upar apne mail ko throw kr rha hoga.
 * - Now if any one db server goes down, there is a loss of data because
 *   linkedlist m jitni nodes thi wo saari lost ho gyi, lekin sbka apna
 *   linkedlist hai.
 * - But the problem is ki ye call avi v fail ho skta hai because first
 *   time iska traffic service-1 m aaya aur wo server down tha to wo
 *   fail krega, aur next retry m sahi chlega kyuki wo request kisi aur
 *   server m jaega.
 * - Second problem is ki agar mera koi server down jaata hai because
 *   it's a stateful server then the data is lost, and mujhe koi idea
 *   nhi hai ki kon kon si email lost ho gyi, because LinkedList mail
 *   k map ko maintain kr rha tha aur wahi lost ho gya. That means jo
 *   2 server chla rhe hai kya wo unko retry kr skte hai? No! Because
 *   inka pta hi nhi hai kon kon sa mail tha. So, the problem basically
 *   occuring is because of a stateful thing, we are very much reliant
 *   on the memory. So, http v yha par ek overhead hai because of its
 *   stateful nature and it's a time consuming task and there is no
 *   acknowledgement, but still a very better solution than the previous
 *   one ki pehle jo hum sync m kr rhe the unhe ab hm async m kr rhe
 *   hai.
 * 
 * Solution: 
 * - Teeno services k LinkedList independently kaam kr rhe hai which is
 *   good thing, but uske kaaran se inke aapas m communicate nhi kr paa
 *   rhe hai.
 * - Saath hi Saath HTTP k hone k kaaran too much traffic or overhead
 *   is coming.
 * - Let's improvise the pattern: Hum ek single server le lenge aur iss
 *   server ka kaam hoga sirf or sirf chijo ko shout krna.
 * - Jab v koi user signup krta hai to can I tell the machine to shout
 *   karo ki ek new user signup hua ya fir send EMAIL to PIYUSH, and
 *   because ye teeno services v same room k andar hai to they will 
 *   listen to it.
 * - Inhone wo event ko listen kiya and based on it wo isko proces kr
 *   skte hai. But isse problem kya solve hua? Isse problem ye solve
 *   hua ki I don't have to deal with ki meri jo request hai wo jis
 *   server k paas route ho rhi hai wo up hai ya nhi. Agar koi server
 *   down hai to listen hi nhi karega, aur humein tension lene ki 
 *   jarurat hi nhi hai ki kisko bheje, humein bas ek shout bhejna hai
 *   aur machine ne listen kiya aur apne apne linkedlist m add kr liya.
 * 
 *                                             +----+
 *                                        +----|    |
 *                                        |    +----+
 *                         SEND EMAIL     |
 *   +----+                 +-------+     |    +----+
 *   |    |---------------->| Shout |<----+----|    |
 *   +----+                 +-------+     |    +----+
 * API Machine                            |
 *                                        |    +----+
 *                                        +----|    |
 *                                             +----+
 *                                          Email Machine
 * Problem:
 * - It's good that we have three copies of the data because sabko mil 
 *   gya, but the problem is ki teeno trigger ho jaenge, jisse 3 emails
 *   chali jaengi.
 * - But isse ye solve ho gya ki ek server down v hai fir v atleast our
 *   data is not getting lost because we have added redundancy so mera
 *   data lost naa ho, but the problem is how to do deduplication?
 * 
 * Solution: Shared Memory
 * - We can use shared memory like Redis.
 * - Agar koi v user kisi particular user ko email bhej rha hai to
 *   pehle uska lock acquire kr lo, aur jab bhej do tab jaa k uska
 *   lock release kr do. Isse hoga ye ki jab ek service ko mail milega
 *   tab wo redis m lock laga dega, aur jab dusre service kov mail
 *   bhejna hoga to wo dekhega ki ispe to lock laga hua hai to wo discard
 *   kr dega, nhi to wo uss email pe lock lga dega.
 * - Isse hoga ki data is deduplicated and 2 baar kabhi send nhi hoga
 *   aur kabhi by chance koi ek server crash ho gya to dusra server k
 *   paas v data hai jisse ye assure hai ki kabhi na kabhi email send
 *   ho hi jaegi.
 * - Aur agar redis crash kr gya to? Isliye hum TTL lagata hai, means
 *   agar 30 min k andar ye mail nhi gya to wo khud hatt jagea aur
 *   retry ho jaega.
 * 
 * 
 * Problem resolved but ye bahut complex ho gya. What is shout service
 * can manage instead of shouting? PubSub - iske andar messages sabko
 * jaate hai but hum nhi chahte the ki messages sabko jae, then we
 * will not use pubsub.
 * 
 * Sol: Let's create an orchestrator/manager jisse hum acchi khasi
 * database de denge. Now we can say ki jo humari API machine hai
 * wo iss manager ko ek message degi, aur ye manager isko apne paas
 * store kr lega. Ab agar ye kabhi crash v hota hai to yha pe data
 * loss nhi ho sakta.
 * 
 * Ab humein ek mechanism banana hai ki ye manager smartly messages
 * ko in workers ko de pae. Lekin manager ko nhi pta kon kon worker
 * hai.
 * 
 * API Machine sirf ek POST call krta hai /publish pe jisse ek message
 * publish ho jaata hai, and manager usse apne database m store kr leta
 * hai. 
 * 
 * Ab agar koi worker up and running aata hai humare system m to wo
 * apni details /REGISTER route pe hit krega. Ab jab ye worker online
 * aae honge to Manager Server pe /REGISTER hit kr k apne aap ko register
 * krwa lenge jisse meri ek aur problem solve ho jaegi that is, now 
 * manager can store their workers in database and manage them. 
 * 
 * Ab iss manager ka simple sa kaam ye hai ki ek while(true) {...} loop
 * chalegi:
 * 
 *   while(true) only process 5m/s
 *     if messages in db
 *     choose a random worker and send to it
 *     delete the message
 * 
 * And because humare haath m control hai to hum bottleneck v laga 
 * sakte hai. But what if manager goes down? Manager is stateless.
 * But what if worker goes down? Maine ek message db se uthata aur
 * ek worker ko de diya aur maine uss message ko delete kr diya aur
 * ye worker uss message ko process kr hi nhi paaya. Technically,
 * manager ne to worker ko de diya but worker ne usse kabhi kiya hi
 * nhi to kya wo message kabhi deliver hua? No!
 * 
 * Solution: Message Broker
 * We should not delete the message, there should be some
 * kind of followup. We should mark ki ek kaam humne kisi worker ko de
 * diya hai to wo kaam kisi aur worker ko nhi dena chahiye and saath
 * hi saath ek time internal v de denge ki manager 10 min baad fir
 * check karega ki kaam hua ki ni:
 * 
 *   while(true)
 *     if messages in db
 *     choose a random worker and send to it
 *     update db = worker_id | TimeInterval 10min
 * 
 * Agar ye message process ho gya to worker wapas manager ko ek ack
 * de sakta hai, uske baad hum message ko delete kr skte hai, but agar
 * 10 min ka interval khatm hua to hum wapas jaa k puch skte hai and
 * based on that agar message process ho gya then that's good but agar
 * process nhi hua to kisi aur worker do denge.
 * 
 *   while(true)
 *     if messages in db
 *     choose a random worker and send to it
 *     update db = assign to new worker (incre retry count)
 * 
 * Ye workers down jaa sakte hai. To kya hum down workers ko kabhi task
 * assign krna chahenge? Jab ye worker aaya hoga to usne khud ko register
 * kr liya but ab wo down hai. To hum ek our route banaenge i.e.,
 * /HEALTH. 
 * 
 * Ab hum apne workers ko bolnge ki har 10sec m apna health check batao.
 * Agar kisi ka 20sec k andar andar health nhi aaya to I will mark you
 * as dead called as LastHeartBeat, aur jaha par workers ka data maintain
 * kr rhe hai waha pe LastHeartBeat time v rakhnge. Ab agar LastHeartBeat
 * time 20s > chali jaati hai to iss worker ko hum kabhi v kuch v
 * assign nhi krnge.
 * 
 * Isi tarah humne khud ka ek Broker Service bana liya jisme we have a
 * manager who acts as a broker who decides ki kon sa message kisse
 * dena hai, kab usse acknowledgement lena hai, aur agar wo nhi kr rha
 * hai to kisi aur ko task denge. And we have build a Push Based Message
 * Broker System.
 * 
 * Ex: RabbitMQ, etc
 * 
 *                                            Workers
 * 
 *                                             +----+
 *                       NodeJS Endpoints +----|    |
 *                        - /PUBLISH      |    +----+
 *                        -               |
 *   +----+      POST       +-------+     |    +----+
 *   |    |---------------->|Manager|<----+----|    |
 *   +----+                 +-------+     |    +----+
 * API Machine                  |         |
 *                              V         |    +----+
 *                          +-------+     +----|    |
 *                          |       |          +----+
 *                          +-------+       Email Machine
 *                           Database
 * 
 * Problem:
 * But as the workers increase as number of messages increase, the
 * manager get overwhelmed because he has a lot of things to manage.
 * Humne manager ko hata diya, aur humne direct database m insert kr
 * diya jaha se Workers khud ba khud task pick kr lenge khud k liye.
 * 
 * while true:
 *   if there is a task
 *   pick it up and perform and delete
 * 
 * But yha pe again we will see the duplication issue.
 * 
 * Sol: 
 * while true:
 *   if there is a task
 *   pick it up 
 *   delete
 *   perform 
 * 
 * Agar task perform hua to success, but agar nhi hua to db m wapas
 * daal do i.e., retry + count. So that dobara se koi na koi worker 
 * perform kr lega.
 * 
 * Problem: But agar perform krte krte khud hi crash ho gya, aur wo
 * usko nhi rakh paya to?
 * 
 * Sol: Don't delete, mark for deletion. Jisse ye pta lg jaega ki ye kaam
 * kisi aur worker ne pickup kr rkha hai. Agar perform ho jae to fir
 * delete kr do with time. Agr ek specific time tak perform nhi hua to
 * kisi aur ko assign kr do. And this mechanism is called Pull based
 * Message Broker.
 * 
 * while true:
 *   if there is a task
 *   pick it up 
 *   mark for deletion + time
 *   perform 
 *   finally delete
 * 
 * Ex: BullMQ
 * 
 * Note: Jab hum queue k saath kaam krte hai to we have two kinds:
 * a. Push based Mechanism
 * b. Pull based Mechanism
 * 
 * But there is a problem. Push based mechanism me do you have to worry
 * about deduplication as a worker? No!. But in Pull based mechanism
 * do you have to worry about deleting the task from the queue, deduplication,
 * acquiring locks? Yes! Because when we are pulling the message we have
 * to get all the locks and stuffs. Agar usme hum lock lagana bhul gye
 * to the problem is our. 
*/


/**
 * Send user realtime updates of Zomato Rider:
 * - Zomato Rider har 500ms m API Service se 
*/

In [ ]:
/**
 *   [Continue.....]
*/

In [ ]:
/**
 * Async Pattern-II
 * 21:27
*/









/**
 * Also note: CQRS and Event Sourcing
 * > ~50:00
*/